# Information about endpoints
THis notebook contains all the information gathered from exploring the service definition & documentation, **available here**: https://aqs.epa.gov/aqsweb/documents/data_api.html
There are also a few functions used in the validation part of the exploration, that I just haven't found the right place for yet.
## Base URL
**base URL** https://aqs.epa.gov/data/api/dailyDate/byCounty?
## Paramaters
email, key, param, bdate, edate, state, county
\
pollutant_code: 42602,88101\
bdate, edate : 20250101 20251231\
state: 29\
county: Saint Louis County: 189, Saint Louis City: 510
### Locations
Paramaters for Missouri, saint louis county\
State: 29\
County: 189
### pollutants
param: 42101,42401,42602,44201,81102,88101

pollutant: Carbon Monoxide, Sulur Dioxide, NO2, Ozone, PM10, PM2.5

Only Nitrogen Dioxide, and Thoracic Fine Particals (PM25):42602,88101 respectively, are measured for both Saint Louis and Saint Louis County.
  
**Note** the EPA excludes Lead, Lead PM10, are excluded from AQI calculations because AQI is a daily measurement, which is at odds with the significant length of time required to effectively collect & analyze lead samples.

## Locations

In [ ]:
url_return_mo_counties="https://aqs.epa.gov/data/api/list/countiesByState?email=test@aqs.api&key=test&state=29"
response_mo_counties_list = requests.get(url_return_mo_counties)

mo_counties=response_mo_counties_list.json()

print(mo_counties)

for row in mo_counties['Data']:
    code = row['code']
    county = row['value_represented']
    if 'Louis' in county:
        print(code)


## Air quality paramaters 
code CRITERIA
value_represented Criteria Pollutants

In [ ]:
url_returns_param_classes="https://aqs.epa.gov/data/api/list/classes?email=test@aqs.api&key=test"
response_param_classes = requests.get(url_returns_param_classes)
print(response_param_classes.status_code)


In [ ]:
# Here you can just get an idea of the categories of parameters available
param_classes=response_param_classes.json()
print(param_classes['Header'])

for key,value in param_classes['Data'][0].items():
    print(key)

for row in param_classes['Data']:
    for key, value in row.items():
        print(key, value)


[{'status': 'Success', 'request_time': '2026-04-16T20:28:13-04:00', 'url': 'https://aqs.epa.gov/data/api/list/classes?email=test@aqs.api&key=test', 'rows': 27}]
code
value_represented
code AIRNOW MAPS
value_represented The parameters represented on AirNow maps (88101, 88502, and 44201)
code ALL
value_represented Select all Parameters Available
code AQI POLLUTANTS
value_represented Pollutants that have an AQI Defined
code CORE_HAPS
value_represented Urban Air Toxic Pollutants
code CRITERIA
value_represented Criteria Pollutants
code CSN DART
value_represented List of CSN speciation parameters to populate the STI DART tool
code FORECAST
value_represented Parameters routinely extracted by AirNow (STI)
code HAPS
value_represented Hazardous Air Pollutants
code IMPROVE CARBON
value_represented IMPROVE Carbon Parameters
code IMPROVE_SPECIATION
value_represented PM2.5 Speciated Parameters Measured at IMPROVE sites
code MET
value_represented Meteorological Parameters
code NATTS CORE HAPS
value_r

In [ ]:
# so you can see the codes for the criteria pollutants
# you could also repurpose this code to view codes for other parameter classes

# This url lets you see the parameters in any of the above classes
url_criteria_pollutants="https://aqs.epa.gov/data/api/list/parametersByClass?email=test@aqs.api&key=test&pc=CRITERIA"
response_criteria_pollutants=requests.get(url_criteria_pollutants)
print(response_criteria_pollutants.status_code)
criteria_pollutants=response_criteria_pollutants.json()

for row in criteria_pollutants['Data']:
    for key, value in row.items():
        print(key, value)


# Plan for Filtering the data
The dictionary available in the pipeline_AQS.py file was used when constructing this plan. The actual exploration isn`t included here, because I reorganized the project, but if you want to do it yourself, after extracting the data, you can use the CreateDataDict class, and pretty easily see why I drew theses conclusions.

## Drop: 
### Info not recorded, duplicate information
* uncertainty- is not recorded
* state, county, date_of_last_change, cbsa_code
* datum
* date_gmt
* time_gmt
* sample_duration_code
### Extranious information
* method,method_type,method_code - not needed for this problem 
* detection_limit - extra detail not needed for this problem
* units_of_measure_code

### Potential outliars that needs inspection
* qualifier - samples with qualifier may need replacement 
* samples with missing sample_measurement need inspection
### Method to reduce samples to one record per criteria per day
* verify for below, that each sample has the same sample_duration
* site_number, parameter, date_local, time_local, poc 
  * sample_measurement take the mean 

## Qualifiers
MNAR -\
Qualifiers indicate the data was corrupted or not collected.
96.122508 have a code ~ 4% with a code
## Decision
These rows will be replaced with the mean value for that paramater.

In [ ]:
# exploring columns with qualifiers
def get_qualifier_info(dataframe):
    info={}
    qualifiers=dataframe.loc[dataframe['qualifier'].notna()]
    info['num_rows']=len(qualifiers)
    info['values']=qualifiers['qualifier'].unique()
    info['df_qualifiers']=qualifiers
    return info 


In [ ]:
aq_hs_df['parameter'].value_counts()


parameter
PM2.5 - Local Conditions    8775
Nitrogen dioxide (NO2)      8530
Name: count, dtype: int64